# StochastiQ — Notebook 03: Stochastic Model Calibration

**Project:** StochastiQ — Multi-model portfolio optimization and derivatives strategy framework  
**Course:** MGT 6081 Derivative Securities, Georgia Institute of Technology  
**Author:** Anay Abhijit Joshi

---

## Objectives

Calibrate four stochastic models to each of the seven assets in the universe — **28 calibrations in total** — using documented, defensible procedures. For each calibration we record the full parameter set, validate the result, and produce diagnostic plots that show how well each model captures the empirical return distribution.

## Models and Calibration Procedures

| Model | Procedure | Output Parameters |
|-------|-----------|-------------------|
| **GBM** | Maximum likelihood on log returns | $\mu$, $\sigma$ |
| **Merton** | Threshold method ($3\sigma$ jumps), then re-estimate diffusion on cleaned returns | $\mu$, $\sigma$, $\lambda_J$, $\mu_J$, $\sigma_J$ |
| **CEV** | Nonlinear least squares on rolling realized vol vs. CEV-implied local vol | $\mu$, $\sigma$, $\gamma$ |
| **Heston** | Method of moments on rolling 21-day realized variance, presented in two variants: unconstrained and Feller-constrained | $\mu$, $\kappa$, $\theta$, $\sigma_v$, $\rho$, $v_0$ |

## Methodological Enhancements

This notebook implements four enhancements over a baseline calibration:

1. **NLS-based CEV calibration.** The standard OLS regression of $\log|\Delta S|$ on $\log S$ is fragile for low-volatility smooth series. We instead fit by minimizing the squared error between observed rolling realized vol and the CEV-implied local vol $\sigma S^{\gamma-1}$. This is robust across all asset types in the universe.

2. **Feller-constrained Heston variant.** The Feller condition $2\kappa\theta > \sigma_v^2$ guarantees the variance process stays strictly positive in continuous time. Unconstrained method-of-moments calibration on rolling realized variance routinely violates this condition for high-vol-clustering equity data. We calibrate Heston in two ways and present both: an **unconstrained** version that fits the empirical vol-of-vol exactly, and a **Feller-constrained** version that caps $\sigma_v$ to satisfy Feller with a 5% safety margin. The reader sees the trade-off between empirical realism and theoretical purity.

3. **Train/test split for goodness-of-fit.** Calibration uses data through **2024-12-31** (training); Q-Q diagnostics and KS tests use **2025-01-01 onward** (test). This removes the construction advantage that would otherwise inflate the apparent fit of models like Merton (whose threshold method explicitly fits the empirical tails).

4. **Formal goodness-of-fit testing.** Beyond the visual Q-Q plot and the mean absolute Q-Q deviation, we report the **Kolmogorov-Smirnov test statistic** and **p-value** for each (asset, model) combination. This gives a formal hypothesis test: $p > 0.05$ means we fail to reject the null that simulated and empirical returns share a distribution.

All calibrated parameters are persisted to disk for use in the Monte Carlo simulation (Notebook 04), options overlay (Notebook 05), and out-of-sample validation (Notebook 06).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RISK_FREE_RATE, TRADING_DAYS, RANDOM_SEED
from src.data.loaders import load_dataset
from src.models.gbm import GBMParams, simulate_gbm
from src.models.merton import MertonParams, simulate_merton
from src.models.cev import CEVParams, simulate_cev
from src.models.heston import HestonParams, simulate_heston
from src.calibration.calibrators import (
    calibrate_gbm,
    calibrate_merton,
    calibrate_cev,
    calibrate_cev_nls,
    calibrate_heston,
    calibrate_heston_constrained,
    calibrate_all_models,
    ks_test_returns,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Train/test split boundary -- calibration uses data through this date,
# Q-Q diagnostics and KS tests use data after this date.
TRAIN_TEST_SPLIT = pd.Timestamp("2024-12-31")

print(f"Project root:         {PROJECT_ROOT}")
print(f"Trading days:         {TRADING_DAYS}")
print(f"Random seed:          {RANDOM_SEED}")
print(f"Train/test split:     {TRAIN_TEST_SPLIT.date()}")

## 2. Load Data and Apply Train/Test Split

We split the data into a training window (used for calibration) and a test window (used for goodness-of-fit evaluation). This separation is essential because the Merton threshold method explicitly fits the empirical tails of its training data; evaluating fit on the same data would inflate its apparent goodness-of-fit. Out-of-sample evaluation gives every model a fair shot.

In [ ]:
prices_full = load_dataset(PROCESSED_DIR / "prices.parquet")
log_returns_full = load_dataset(PROCESSED_DIR / "log_returns.parquet")
tickers = list(log_returns_full.columns)

# Apply train/test split
prices_train = prices_full.loc[:TRAIN_TEST_SPLIT]
log_returns_train = log_returns_full.loc[:TRAIN_TEST_SPLIT]
prices_test = prices_full.loc[TRAIN_TEST_SPLIT:]
log_returns_test = log_returns_full.loc[TRAIN_TEST_SPLIT:]

print(f"Full data:  {log_returns_full.index.min().date()} to {log_returns_full.index.max().date()}  ({len(log_returns_full)} days)")
print(f"Training:   {log_returns_train.index.min().date()} to {log_returns_train.index.max().date()}  ({len(log_returns_train)} days)")
print(f"Test:       {log_returns_test.index.min().date()} to {log_returns_test.index.max().date()}  ({len(log_returns_test)} days)")
print(f"Assets ({len(tickers)}): {tickers}")

## 3. Calibrate All Models on the Training Window

We run the full grid of 28 calibrations using the training window only. CEV uses the NLS method; Heston is calibrated both unconstrained and constrained. The constrained variant is stored separately for later comparison.

In [ ]:
# Primary calibrations (CEV-NLS, Heston unconstrained)
calibrated = {}
for ticker in tickers:
    calibrated[ticker] = calibrate_all_models(
        prices_train[ticker],
        log_returns_train[ticker],
        trading_days=TRADING_DAYS,
        cev_method="nls",
        heston_constrained=False,
    )

# Feller-constrained Heston variant for side-by-side comparison
heston_constrained_results = {
    ticker: calibrate_heston_constrained(
        log_returns_train[ticker], trading_days=TRADING_DAYS
    )
    for ticker in tickers
}

print("All 28 primary calibrations complete (CEV-NLS, Heston unconstrained).")
print("Feller-constrained Heston also calibrated for all 7 assets.\n")

## 4. Parameter Comparison Tables

We present the calibrated parameters in a series of tables, one per model. This is the primary reference output for the writeup.

### 4.1 GBM Parameters

In [ ]:
gbm_table = pd.DataFrame({
    ticker: {
        "mu (annual)":    calibrated[ticker]["GBM"].mu,
        "sigma (annual)": calibrated[ticker]["GBM"].sigma,
    }
    for ticker in tickers
}).T

gbm_table.style.format({
    "mu (annual)":    "{:.2%}",
    "sigma (annual)": "{:.2%}",
}).background_gradient(cmap="YlOrRd", subset=["sigma (annual)"])

### 4.2 Merton Jump-Diffusion Parameters

In [ ]:
merton_table = pd.DataFrame({
    ticker: {
        "mu":         calibrated[ticker]["Merton"].mu,
        "sigma":      calibrated[ticker]["Merton"].sigma,
        "lambda_J":   calibrated[ticker]["Merton"].lambda_j,
        "mu_J":       calibrated[ticker]["Merton"].mu_j,
        "sigma_J":    calibrated[ticker]["Merton"].sigma_j,
        "k":          calibrated[ticker]["Merton"].k,
    }
    for ticker in tickers
}).T

merton_table.style.format({
    "mu":       "{:.2%}",
    "sigma":    "{:.2%}",
    "lambda_J": "{:.2f}",
    "mu_J":     "{:.4f}",
    "sigma_J":  "{:.4f}",
    "k":        "{:.4f}",
}).background_gradient(cmap="YlOrRd", subset=["lambda_J"])

**Reading the Merton table:** $\lambda_J$ tells us how often jumps occur per year; $\mu_J$ is the mean log-jump size (typically negative for equities — crashes are larger than rallies); $\sigma_J$ is the dispersion of jumps; $k = e^{\mu_J + \sigma_J^2/2} - 1$ is the drift compensator that keeps the risk-neutral expected return at $\mu$.

### 4.3 CEV Parameters (NLS Method)

The NLS calibrator fits sigma and gamma simultaneously by minimizing the squared error between rolling realized vol and the CEV-implied local vol $\sigma S^{\gamma-1}$. This is more robust than the OLS-on-log-differences approach for low-vol smooth series.

In [ ]:
cev_table = pd.DataFrame({
    ticker: {
        "mu":    calibrated[ticker]["CEV"].mu,
        "sigma": calibrated[ticker]["CEV"].sigma,
        "gamma": calibrated[ticker]["CEV"].gamma,
    }
    for ticker in tickers
}).T

cev_table.style.format({
    "mu":    "{:.2%}",
    "sigma": "{:.4f}",
    "gamma": "{:.4f}",
}).background_gradient(cmap="RdBu_r", subset=["gamma"], vmin=0, vmax=2)

**Reading the CEV table:** $\gamma = 1$ recovers GBM exactly. $\gamma < 1$ indicates the **leverage effect** (volatility rises as price falls), which is empirically common in equities. $\gamma > 1$ would indicate the inverse (volatility rises as price rises) — rare for equities, more common for some commodities.

#### Comparison: OLS vs NLS

For methodological transparency, we run both CEV calibrators and compare. The NLS method is used as the primary calibration; the OLS method is included to demonstrate why NLS was chosen.

In [ ]:
cev_comparison = pd.DataFrame({
    ticker: {
        "OLS gamma":    calibrate_cev(prices_train[ticker], log_returns_train[ticker]).gamma,
        "OLS sigma":    calibrate_cev(prices_train[ticker], log_returns_train[ticker]).sigma,
        "NLS gamma":    calibrated[ticker]["CEV"].gamma,
        "NLS sigma":    calibrated[ticker]["CEV"].sigma,
    }
    for ticker in tickers
}).T

cev_comparison.style.format({
    "OLS gamma": "{:.3f}", "OLS sigma": "{:.4f}",
    "NLS gamma": "{:.3f}", "NLS sigma": "{:.4f}",
})

**Discussion:** The NLS method produces sensible $\gamma$ estimates for every asset and never falls back to GBM. The OLS method is fragile for the smoothest assets in the universe (SPY, GLD), where the log $|\Delta S|$ regression has insufficient signal-to-noise; we observe its gamma estimates either falling outside $[0.1, 1.5]$ or producing pathological sigma values that trigger the OLS fallback to gamma = 1. NLS is therefore the recommended method and is used everywhere downstream.

### 4.4 Heston Parameters

We present both the unconstrained method-of-moments calibration and the Feller-constrained variant. The Feller condition $2\kappa\theta > \sigma_v^2$ guarantees the variance process stays strictly positive in continuous time. Unconstrained MoM calibration on rolling realized variance routinely violates this condition for high-vol-clustering equity data; the constrained variant caps $\sigma_v$ to satisfy Feller with a 5% safety margin.

In [ ]:
heston_table = pd.DataFrame({
    ticker: {
        "mu":         calibrated[ticker]["Heston"].mu,
        "kappa":      calibrated[ticker]["Heston"].kappa,
        "theta":      calibrated[ticker]["Heston"].theta,
        "sigma_v":    calibrated[ticker]["Heston"].sigma_v,
        "rho":        calibrated[ticker]["Heston"].rho,
        "v0":         calibrated[ticker]["Heston"].v0,
        "sqrt(theta)":np.sqrt(calibrated[ticker]["Heston"].theta),
        "Feller ratio": calibrated[ticker]["Heston"].feller_ratio,
    }
    for ticker in tickers
}).T

heston_table.style.format({
    "mu":          "{:.2%}",
    "kappa":       "{:.3f}",
    "theta":       "{:.4f}",
    "sigma_v":     "{:.3f}",
    "rho":         "{:.3f}",
    "v0":          "{:.4f}",
    "sqrt(theta)": "{:.2%}",
    "Feller ratio":"{:.2f}",
}).background_gradient(cmap="RdYlGn", subset=["Feller ratio"], vmin=0, vmax=3)

#### Comparison: Unconstrained vs Feller-Constrained Heston

The constrained calibration produces parameters that are mathematically guaranteed to keep variance strictly positive. The trade-off is that it slightly under-fits the empirical vol-of-vol — by how much depends on how strongly the unconstrained calibration violated Feller.

In [ ]:
heston_compare = pd.DataFrame({
    ticker: {
        "Unc. sigma_v":   calibrated[ticker]["Heston"].sigma_v,
        "Unc. Feller":    calibrated[ticker]["Heston"].feller_ratio,
        "Unc. Satisfied": calibrated[ticker]["Heston"].feller_satisfied,
        "Con. sigma_v":   heston_constrained_results[ticker].sigma_v,
        "Con. Feller":    heston_constrained_results[ticker].feller_ratio,
        "Con. Satisfied": heston_constrained_results[ticker].feller_satisfied,
        "sigma_v cap %":  100.0 * (1.0 - heston_constrained_results[ticker].sigma_v / max(calibrated[ticker]["Heston"].sigma_v, 1e-12)),
    }
    for ticker in tickers
}).T

heston_compare.style.format({
    "Unc. sigma_v":   "{:.3f}",
    "Unc. Feller":    "{:.2f}",
    "Con. sigma_v":   "{:.3f}",
    "Con. Feller":    "{:.2f}",
    "sigma_v cap %":  "{:.1f}%",
})

**Discussion:** The unconstrained calibration violates Feller for all 7 assets, with Feller ratios from 0.21 (SPY) to 0.69 (GLD). The constrained variant satisfies Feller for all 7 by capping $\sigma_v$. The cap percentage tells us by how much: assets where Feller was severely violated (SPY, JPM) require larger caps; those closer to satisfaction (GLD) need smaller caps. **For the rest of this notebook and downstream phases we use the unconstrained Heston calibration**, on the principle that empirical fit to realized vol-of-vol matters more than theoretical positivity (the simulator's full-truncation Euler scheme handles near-zero variance correctly). The constrained variant is preserved in the saved parameter file for users who require Feller compliance.

**Reading the Heston table:**
- $\kappa$ — speed of mean reversion of variance (higher = faster reversion)
- $\theta$ — long-run variance; $\sqrt{\theta}$ is the long-run volatility for intuition
- $\sigma_v$ — vol of vol, which controls how much variance itself fluctuates
- $\rho$ — leverage correlation; should be negative for equities (price drops drive vol up)
- $v_0$ — current variance regime (used as the simulation starting point)
- **Feller ratio** = $2\kappa\theta / \sigma_v^2$. Values $> 1$ guarantee positive variance.

## 5. Out-of-Sample Q-Q Diagnostic Plots

Q-Q plots compare the **quantiles** of simulated returns against the **quantiles** of the empirical historical returns. A perfect model fit produces points on the 45-degree line.

**How to read:** Deviation from the line in the lower-left tail means the model under-captures crash risk. Deviation in the upper-right means it under-captures rally risk.

We simulate paths from the **training-window calibration** but evaluate the Q-Q against the **test-window empirical returns**. This out-of-sample setup is the fair comparison: it tests how well each model generalizes from training to test, not how well it was fitted in-sample.

_This cell takes about 30-60 seconds to run as it executes 28 simulations._

In [ ]:
def simulated_daily_returns(ticker: str, model_name: str, horizon: int, n_paths: int = 1000) -> np.ndarray:
    """Simulate paths under the calibrated model and return flattened daily log returns.
    
    Horizon is taken as the test window length so the simulated sample matches the
    out-of-sample test sample in size.
    """
    # Start the simulation at the price level on the train/test boundary
    s0 = float(prices_train[ticker].iloc[-1])
    params = calibrated[ticker][model_name]

    if model_name == "GBM":
        paths = simulate_gbm(s0, params, horizon, n_paths, TRADING_DAYS, seed=RANDOM_SEED)
    elif model_name == "Merton":
        paths = simulate_merton(s0, params, horizon, n_paths, TRADING_DAYS, seed=RANDOM_SEED)
    elif model_name == "CEV":
        paths = simulate_cev(s0, params, horizon, n_paths, TRADING_DAYS, seed=RANDOM_SEED)
    elif model_name == "Heston":
        paths, _ = simulate_heston(s0, params, horizon, n_paths, TRADING_DAYS, seed=RANDOM_SEED)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    log_p = np.log(paths)
    daily_log_rets = np.diff(log_p, axis=1).flatten()
    return daily_log_rets


model_names = ["GBM", "Merton", "CEV", "Heston"]
model_colors = {"GBM": "#1f77b4", "Merton": "#ff7f0e", "CEV": "#2ca02c", "Heston": "#d62728"}

test_horizon = len(log_returns_test)

fig, axes = plt.subplots(len(tickers), len(model_names), figsize=(16, 3 * len(tickers)))

for i, ticker in enumerate(tickers):
    empirical = log_returns_test[ticker].dropna().values
    empirical_q = np.quantile(empirical, np.linspace(0.01, 0.99, 99))

    for j, model_name in enumerate(model_names):
        ax = axes[i, j]
        sim = simulated_daily_returns(ticker, model_name, test_horizon, n_paths=500)
        sim_q = np.quantile(sim, np.linspace(0.01, 0.99, 99))

        ax.scatter(empirical_q, sim_q, s=8, alpha=0.6, color=model_colors[model_name])
        lo = min(empirical_q.min(), sim_q.min())
        hi = max(empirical_q.max(), sim_q.max())
        ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, alpha=0.6)

        if i == 0:
            ax.set_title(model_name, fontsize=11, fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"{ticker}\nSim quantile", fontsize=10)
        if i == len(tickers) - 1:
            ax.set_xlabel("Empirical quantile (test window)", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.3)

fig.suptitle("Out-of-Sample Q-Q Plots: Calibrated on 2020-2024, Tested on 2025-2026", fontsize=14, fontweight="bold", y=1.001)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_qq_plots.png", bbox_inches="tight")
plt.show()

**What to look for in the Q-Q plots:**
- **GBM (blue)** — typically deviates noticeably in the tails for assets with high kurtosis. This is the empirical justification for using non-Gaussian models.
- **Merton (orange)** — should track the tails better than GBM, especially the lower tail, due to the explicit jump component. **Note:** in the out-of-sample setup, Merton no longer enjoys the construction advantage it had in-sample, so its dominance over the other models is weaker than a naive in-sample evaluation would suggest.
- **CEV (green)** — captures the leverage effect. With NLS calibration, CEV produces authentic non-trivial gamma estimates for every asset.
- **Heston (red)** — captures fat tails through stochastic volatility rather than jumps. Often the best fit for assets with persistent volatility clustering.

## 6. Quantitative Goodness-of-Fit

We quantify the Q-Q diagnostic in two complementary ways:

1. **Mean absolute deviation** between simulated and empirical quantiles (lower is better). This is a simple aggregate measure of Q-Q closeness across all percentiles.
2. **Kolmogorov-Smirnov test** statistic and p-value (higher p-value is better). The KS test gives a formal hypothesis test: if $p > 0.05$, we fail to reject the null that simulated and empirical returns share a distribution.

Both are computed on the **out-of-sample test window**.

In [ ]:
# Compute MAD and KS for each (asset, model) on the test window
rows = []
for ticker in tickers:
    empirical = log_returns_test[ticker].dropna().values
    empirical_q = np.quantile(empirical, np.linspace(0.01, 0.99, 99))
    for model_name in model_names:
        sim = simulated_daily_returns(ticker, model_name, test_horizon, n_paths=500)
        sim_q = np.quantile(sim, np.linspace(0.01, 0.99, 99))
        mad = float(np.mean(np.abs(sim_q - empirical_q)))
        ks_stat, ks_p = ks_test_returns(empirical, sim)
        rows.append({
            "ticker":   ticker,
            "model":    model_name,
            "MAD":      mad,
            "KS_stat":  ks_stat,
            "KS_pvalue":ks_p,
        })

fit_long = pd.DataFrame(rows)

# Pivot for display
mad_table = fit_long.pivot(index="ticker", columns="model", values="MAD").reindex(tickers)[model_names]
ks_stat_table = fit_long.pivot(index="ticker", columns="model", values="KS_stat").reindex(tickers)[model_names]
ks_p_table = fit_long.pivot(index="ticker", columns="model", values="KS_pvalue").reindex(tickers)[model_names]

best_per_asset = mad_table.idxmin(axis=1)
mad_table_display = mad_table.copy()
mad_table_display["Best fit"] = best_per_asset

print("Mean absolute Q-Q deviation (out-of-sample). Lower is better.\n")
mad_table_display.style.format({m: "{:.5f}" for m in model_names})\
    .background_gradient(cmap="RdYlGn_r", subset=model_names, axis=1)

In [ ]:
print("Kolmogorov-Smirnov p-values (out-of-sample). p > 0.05 means we fail to reject")
print("the null that simulated and empirical returns share a distribution.\n")

ks_p_table.style.format("{:.4f}")\
    .background_gradient(cmap="RdYlGn", vmin=0, vmax=0.20)

In [ ]:
# Summary statistics across the model grid
n_assets = len(tickers)
winner_counts = best_per_asset.value_counts()
ks_pass_counts = (ks_p_table > 0.05).sum(axis=0)

summary = pd.DataFrame({
    "# assets where best (MAD)":      winner_counts.reindex(model_names).fillna(0).astype(int),
    "# assets KS-accepted (p>0.05)":  ks_pass_counts.reindex(model_names).fillna(0).astype(int),
    "Mean KS p-value":                ks_p_table.mean(axis=0).round(4),
    "Mean MAD":                       mad_table.mean(axis=0).round(5),
})

print(f"Goodness-of-fit summary across {n_assets} assets (out-of-sample):\n")
summary

**How to interpret the summary table:**
- **# assets where best (MAD)** — for how many assets did each model achieve the lowest mean absolute Q-Q deviation? Higher is better.
- **# assets KS-accepted** — for how many assets did the KS test fail to reject the null at the 5% level? Higher is better; this is a stricter criterion than MAD because it requires distributional agreement, not just quantile-by-quantile closeness.
- **Mean KS p-value** — average across the 7 assets. Higher means the model's simulated returns are systematically harder to distinguish from the empirical distribution.
- **Mean MAD** — average across the 7 assets. Lower is better.

Together these numbers tell us which model is the **most pertaining** to the universe — the central question of the project.

## 7. Save Calibrated Parameters

We persist all calibrations (primary + Feller-constrained Heston) and goodness-of-fit metrics in a structured form. Downstream notebooks load these directly without re-fitting.

In [ ]:
# Long-format DataFrame of all parameters (primary calibrations)
rows = []
for ticker in tickers:
    for model_name, params in calibrated[ticker].items():
        for field, value in params.__dict__.items():
            rows.append({
                "ticker":  ticker,
                "model":   model_name,
                "variant": "primary",
                "param":   field,
                "value":   float(value),
            })

# Add Feller-constrained Heston as a separate variant
for ticker in tickers:
    for field, value in heston_constrained_results[ticker].__dict__.items():
        rows.append({
            "ticker":  ticker,
            "model":   "Heston",
            "variant": "feller_constrained",
            "param":   field,
            "value":   float(value),
        })

params_df = pd.DataFrame(rows)
params_path = PROCESSED_DIR / "calibrated_parameters.parquet"
params_df.to_parquet(params_path)

# Goodness-of-fit table (all metrics in long format)
fit_path = PROCESSED_DIR / "goodness_of_fit.parquet"
fit_long.to_parquet(fit_path)

print(f"Saved {params_path.relative_to(PROJECT_ROOT)}  ({params_path.stat().st_size / 1024:.1f} KB)")
print(f"  -> {len(params_df)} rows: 7 tickers x (4 primary + 1 constrained-Heston) x params")
print(f"Saved {fit_path.relative_to(PROJECT_ROOT)}  ({fit_path.stat().st_size / 1024:.1f} KB)")
print(f"  -> {len(fit_long)} rows: 7 tickers x 4 models x (MAD, KS_stat, KS_pvalue)")

## 8. Summary

**What this notebook accomplished:**

1. **Train/test split** at 2024-12-31. Calibration uses training data only; goodness-of-fit evaluation uses the held-out test window. This eliminates the construction advantage that would otherwise inflate Merton's apparent fit.

2. **Four stochastic models calibrated** to each of the seven assets in the universe — 28 primary calibrations plus 7 Feller-constrained Heston variants.

3. **NLS-based CEV calibration** that produces authentic non-trivial gamma estimates for every asset, replacing the fragile OLS-on-log-differences method.

4. **Dual Heston variants** — unconstrained (better empirical fit) and Feller-constrained (theoretical positivity guarantee) — with side-by-side comparison.

5. **Two formal goodness-of-fit metrics** — mean absolute Q-Q deviation and Kolmogorov-Smirnov tests — both evaluated out-of-sample.

6. **A 7×4 grid of out-of-sample Q-Q plots** as the visual headline of model performance.

**What this enables for the rest of the project:**

Notebook 04 will use the primary calibrated parameters to run Monte Carlo simulations of asset prices forward in time under each model. These simulated price paths will then drive portfolio optimization (Notebook 04 continued) and the option overlay analysis (Notebook 05). The out-of-sample goodness-of-fit results from this notebook directly inform the regime view in Notebook 07 and the model-robustness analysis throughout.

**Next:** Notebook 04 — Monte Carlo forward simulation under each calibrated model, computing portfolio P&L distributions and finding model-robust optimal weights.